# Practice: the fitted truth start

A sandbox for the start rule in `orbit_correction.truth_prop`. Truth is started from an
SGP4 element set. The epoch-state rule (`preserve_sgp4_mean_motion`, the default until
2026-11-25) starts from SGP4's state at the epoch. That state carries SGP4's short-periodic
terms, so its osculating mean motion depends on **where in its orbit the object is at the
epoch**, and truth drifts along-track against SGP4 by an amount that changes sign with the
orbital phase. The fitted rule (`fit_to_sgp4_before_epoch`, the configured default since
2026-11-25, with a 0.25-day span) fits the epoch state to SGP4 positions over a span
*before* the epoch, which averages the short-periodic terms out.

What you can change here: the eccentricity, the mean anomaly at the epoch (the phase), the
fit span, and (in the last section) where the fit span sits. Each run is a 7-day truth
propagation in this kernel's JVM, about 1-3 s.

Inputs: the committed fixtures (`tests/fixtures/celestrak_gp_sample.json`, NORAD 25544's
element set as a template; `tests/fixtures/sw_celestrak_2024_extract.txt`). B* is set to
1e-7 so drag is negligible and only the start differs. Nothing here writes to `src/` or
`data/`. Needs `uv sync --group datagen`.

In [ ]:
import dataclasses
import json
import math
import tempfile
from pathlib import Path

import numpy as np

from orbit_correction import truth_prop as T
from orbit_correction.cheap_prop import propagate_sgp4_from_epoch, sgp4_epoch_states
from orbit_correction.frames import utc_to_tt_seconds
from orbit_correction.residuals import rtn_from_states
from orbit_correction.types import Trajectory

REPO = T.REPO
DAY = 86400.0
EPOCH = "2024-06-15T00:00:00"
FIX = REPO / "tests" / "fixtures"
TEMPLATE = next(r for r in json.loads((FIX / "celestrak_gp_sample.json").read_text())
                if int(r["NORAD_CAT_ID"]) == 25544)
TMP = Path(tempfile.mkdtemp(prefix="fitted_start_"))
SW = T.write_space_weather_copy(FIX / "sw_celestrak_2024_extract.txt", TMP / "sw.txt",
                                utc_to_tt_seconds("2024-06-10T00:00:00"),
                                utc_to_tt_seconds("2024-06-23T00:00:00"))
BASE = T.load_truth_config()
ctx = T._context(BASE)  # starts this kernel's JVM (once)
print("configured rule:", BASE.start_rule, "| fit span", BASE.fit_span_s / DAY, "d, step",
      BASE.fit_step_s, "s")

## One window

`elset(e, ma)` takes the template, sets perigee 800 km, eccentricity `e`, mean anomaly `ma`
(degrees) and B* = 1e-7. `run(...)` propagates truth for 7 days under a rule and returns SGP4
minus truth in curvilinear RTN (km), plus the fit's record for the fitted rule.

In [ ]:
def elset(e=0.001, ma=90.0, hp_km=800.0, argp=None, bstar=1e-7):
    rec = dict(TEMPLATE)
    a = (6378.135 + hp_km) / (1 - e)  # WGS-72 radius and GM: SGP4's mean elements
    rec.update(ECCENTRICITY=e, MEAN_ANOMALY=ma, BSTAR=bstar,
               MEAN_MOTION=math.sqrt(398600.8 / a**3) * DAY / (2 * math.pi))
    if argp is not None:
        rec["ARG_OF_PERICENTER"] = argp
    return T._omm_record_elset(rec, EPOCH)


DT = 600.0 * np.arange(7 * 144 + 1)  # 7 days, 10 min


def run(el, rule=T.EPOCH_START_RULE, span_days=None, samples=None):
    # every run names its rule; span_days=None keeps the configured span (0.25 d)
    row = el.to_pylist()[0]
    span_s = BASE.fit_span_s if span_days is None else span_days * DAY
    cfg = dataclasses.replace(BASE, start_rule=rule, fit_span_s=span_s)
    init = sgp4_epoch_states(el)[0].trajectory
    kw = {}
    if rule == T.FIT_START_RULE:
        kw["fit_samples"] = samples if samples is not None else (
            T.fit_samples_from_elsets(el, cfg)[0].trajectory)
    res = T.propagate_truth(init, row["bstar_per_earth_radii"], row["epoch_tt_s"] + DT, cfg,
                            SW, **kw)
    sg = propagate_sgp4_from_epoch(el, DT)[0].trajectory
    rtn = rtn_from_states(sg.r_gcrf_km, sg.v_gcrf_km_s, res.trajectory.r_gcrf_km)
    return rtn, res.start_fit


rtn, _ = run(elset(0.001, 90.0))
rtn_f, fit = run(elset(0.001, 90.0), T.FIT_START_RULE)
for h, j in (("1 d", 144), ("3 d", 432), ("7 d", 1008)):
    print(f"T at {h}: epoch start {rtn[j, 1]:+8.3f} km | fitted start {rtn_f[j, 1]:+8.3f} km")
print(f"fit: {fit.status} in {fit.iterations} passes, RMS {fit.rms_initial_m:.0f} m -> "
      f"{fit.rms_m:.1f} m over {fit.n_samples} SGP4 positions")

## Experiment 1: sweep the phase

The same orbit, started at eight mean anomalies. Under the epoch start the 7-day T swings
with the phase (roughly a sine in the mean anomaly, bigger for larger `e`). Under the fitted
start it should stay near zero. Try `E = 0.07`, or set `argp=0.0` in `elset`.

In [ ]:
E = 0.02
print(f"e = {E}:  MA   T7d epoch start   T7d fitted start   fit RMS (m)")
for ma in range(0, 360, 45):
    a, _ = run(elset(E, float(ma)))
    b, f = run(elset(E, float(ma)), T.FIT_START_RULE)
    print(f"        {ma:3d}   {a[-1, 1]:+12.2f}       {b[-1, 1]:+12.3f}        {f.rms_m:8.1f}")

## Experiment 2: change the span

The span sets how many revolutions the fit averages over. A span much shorter than one
revolution (about 100 minutes here) cannot average the short-periodic terms; longer spans
average more of them but ask the truth model to match SGP4 further back.

In [ ]:
el = elset(0.02, 270.0)
for span in (0.05, 0.1, 0.25, 0.5, 1.0, 2.0):
    b, f = run(el, T.FIT_START_RULE, span_days=span)
    print(f"span {span:5.2f} d: {f.n_samples:5d} samples, RMS {f.rms_m:7.1f} m, "
          f"T7d {b[-1, 1]:+8.3f} km, {f.iterations} passes, {f.wall_s:.1f} s")

## Experiment 3: why the span sits before the epoch

The project's fit refuses any SGP4 state after the epoch. The first cell shows the refusal.
The second swaps in two **notebook-only** copies: a check that lets later samples through,
and a fit pass that integrates forwards from the epoch instead of backwards. It then fits
over `[epoch, epoch + 1 d]`. That start has seen SGP4 inside the prediction window, so the
residual over the first day shrinks for a reason no model input can know about: leakage.
The cell changes nothing outside this kernel and restores both functions.

In [ ]:
# a 400 km orbit with drag, where SGP4 and truth diverge over the first day
el = elset(0.001, 90.0, hp_km=400.0, bstar=3e-4)
init = sgp4_epoch_states(el)[0].trajectory
after = propagate_sgp4_from_epoch(el, np.arange(0.0, DAY + 1, 60.0))[0].trajectory
try:
    T.fit_start_to_sgp4(init, after, 3e-4, dataclasses.replace(BASE, start_rule=T.FIT_START_RULE),
                        SW)
except ValueError as exc:
    print("refused:", exc)

In [ ]:
def forward_fit_pass(cfg, ctx, x_m, t0, dates, a2m, space_weather):
    # a copy of truth_prop._fit_pass that integrates forwards to the last sample
    state = Trajectory(1, np.array([t0]), x_m[None, :3] / 1000.0, x_m[None, 3:] / 1000.0, "fit")
    prop, integ, _ = T.build_propagator(cfg, state, a2m, space_weather)
    harvester = prop.setupMatricesComputation("stm", None, None)
    generator = prop.getEphemerisGenerator()
    prop.propagate(ctx.date(t0), dates[-1])
    ephemeris = generator.getGeneratedEphemeris()
    pos, jac = np.empty((len(dates), 3)), np.empty((len(dates), 3, 6))
    for j, d in enumerate(dates):
        s = ephemeris.propagate(d)
        p = s.getPVCoordinates(ctx.gcrf).getPosition()
        pos[j] = (p.getX(), p.getY(), p.getZ())
        phi = harvester.getStateTransitionMatrix(s)
        jac[j] = [[phi.getEntry(a, b) for b in range(6)] for a in range(3)]
    return pos, jac, int(integ.getEvaluations())


real_check, real_pass = T._check_fit_samples, T._fit_pass
T._check_fit_samples = lambda initial, samples: None  # this kernel only
T._fit_pass = forward_fit_pass
try:
    leaky, f = run(el, T.FIT_START_RULE, samples=after)
finally:
    T._check_fit_samples, T._fit_pass = real_check, real_pass
honest, _ = run(el, T.FIT_START_RULE, span_days=1.0)  # as long as the leaky span
for h, j in (("6 h", 36), ("1 d", 144), ("7 d", 1008)):
    print(f"|dr| at {h}: span before the epoch {np.linalg.norm(honest[j]):7.3f} km | "
          f"span after the epoch {np.linalg.norm(leaky[j]):7.3f} km")

## Questions to try

- At which eccentricity does the epoch start's phase swing pass 100 km at 7 days?
- Does the fitted start's 7-day T depend on the span once the span is longer than a few
  revolutions?
- With drag on (`bstar=1e-4` at 400 km), how much of the 7-day T does the start rule still
  explain?